In [42]:
import torch
from matplotlib.pyplot import pie
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
import numpy as np
import pandas as pd


In [43]:
pixels_list = []
df = pd.read_csv('fer2013.csv')
pixels_series = df['pixels'].values
for pixels in pixels_series:
    pixels_values = [int(p) for p in pixels.split()]
    pixels_list.append(pixels_values)

pixels_array = np.array(pixels_list, dtype=np.uint8)
pixels_tensor = torch.from_numpy(pixels_array)

batch_size = pixels_tensor.shape[0]
images_tensor = pixels_tensor.view(batch_size, 48, 48)
images_tensor = images_tensor / 255.0
images_tensor

tensor([[[0.2745, 0.3137, 0.3216,  ..., 0.2039, 0.1686, 0.1608],
         [0.2549, 0.2392, 0.2275,  ..., 0.2196, 0.2039, 0.1725],
         [0.1961, 0.1686, 0.2118,  ..., 0.1922, 0.2196, 0.1843],
         ...,
         [0.3569, 0.2549, 0.1647,  ..., 0.2824, 0.2196, 0.1686],
         [0.3020, 0.3216, 0.3098,  ..., 0.4118, 0.2745, 0.1804],
         [0.3020, 0.2824, 0.3294,  ..., 0.4157, 0.4275, 0.3216]],

        [[0.5922, 0.5882, 0.5765,  ..., 0.5059, 0.5490, 0.4706],
         [0.5922, 0.5843, 0.5843,  ..., 0.4784, 0.5529, 0.5373],
         [0.5922, 0.5922, 0.6118,  ..., 0.4275, 0.4824, 0.5725],
         ...,
         [0.7373, 0.7373, 0.4745,  ..., 0.7255, 0.7255, 0.7294],
         [0.7373, 0.7333, 0.7686,  ..., 0.7294, 0.7137, 0.7333],
         [0.7294, 0.7216, 0.7255,  ..., 0.7569, 0.7176, 0.7216]],

        [[0.9059, 0.8314, 0.6118,  ..., 0.1725, 0.1059, 0.0627],
         [0.8980, 0.6863, 0.5804,  ..., 0.1059, 0.1373, 0.1059],
         [0.8392, 0.6118, 0.6157,  ..., 0.1098, 0.0863, 0.

In [44]:
images_tensor[1].shape

torch.Size([48, 48])

### 缓解过拟合操作一：引入数据增强

In [45]:
from torchvision import transforms
from torchvision.transforms import ToPILImage, ToTensor

transform_pipeline = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),                # 随机水平翻转
    transforms.RandomRotation(degrees=15),                 # 随机旋转 ±15度
    transforms.RandomResizedCrop(size=(48,48), scale=(0.8,1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 轻微颜色抖动
    transforms.ToTensor(),                                  # 转张量并缩放到[0,1]
    transforms.Normalize(mean=[0.5], std=[0.5])             # 标准化到[-1,1]，可选
])

to_pil = ToPILImage()
to_tensor = ToTensor()
transformed_images = []
for image in images_tensor:  # image.shape 为 [48, 48]
    if image.dim() == 2:
        image = image.unsqueeze(0)  # 现在 shape 是 [1, 48, 48]

    img_pil = to_pil(image)

    # 应用变换
    transformed_img = transform_pipeline(img_pil)
    transformed_images.append(transformed_img)

In [46]:
transformed_images[0].shape

torch.Size([1, 48, 48])

In [47]:
from torch.utils.data import DataLoader, random_split, TensorDataset
import torch

if isinstance(transformed_images, list):
    # 将列表中的图像堆叠成一个张量
    X = torch.stack(transformed_images, dim=0)
else:
    X = transformed_images
y = torch.tensor(df['emotion'], dtype=torch.long)  # 标签张量

# 创建图像-标签配对的数据集
dataset = TensorDataset(X, y)

# 设置划分比例
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15  # 总和为 1

# 计算各集大小
dataset_size = len(dataset)
train_size = int(train_ratio * dataset_size)
val_size = int(val_ratio * dataset_size)
test_size = dataset_size - train_size - val_size  # 处理舍入误差

# 随机划分数据集
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 32

# 创建数据加载器
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,        # 仅训练集打乱
    num_workers=2,
    pin_memory=True      # GPU 加速
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("数据集划分：")
print(f"训练集: {len(train_dataset)} 个样本")
print(f"验证集: {len(val_dataset)} 个样本")
print(f"测试集: {len(test_dataset)} 个样本")

数据集划分：
训练集: 25120 个样本
验证集: 5383 个样本
测试集: 5384 个样本


In [48]:
df['emotion'].unique()

array([0, 2, 4, 6, 3, 5, 1])

In [49]:
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_rate=0.7):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout_rate) if dropout_rate > 0 else nn.Identity()

        # 捷径连接：当输入输出维度不匹配时，需要1x1卷积调整
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = x

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.dropout(out)
        # 应用捷径连接
        out += self.shortcut(residual)
        out = F.relu(out)

        return out

### 缓解过拟合二：引入dropout层『初始为：0.5，效果不明显改为：0.7』

In [50]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=7, dropout_rate=0.7):
        super(ResNet, self).__init__()
        self.in_channels = 64

        # 初始卷积层
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # 对于灰度图像，输入通道为1
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # 四个残差块层
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1 )
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)  # 下采样
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # 全局平均池化和全连接层

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        in_channels = self.in_channels

        for stride in strides:
            layers.append(block(in_channels, out_channels, stride))
            in_channels = out_channels

        self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.dropout(out)
        out = self.fc(out)

        return out

In [51]:
def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=7)


In [52]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet18().to(device)

### 缓解方法三：增大权重衰减

In [53]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=100)

In [54]:
def model_train(model, device, train_loader,val_loader, optimizer, scheduler, epochs):
    best_acc = 0.0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            output = model(inputs)
            loss = loss_fn(output, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = 100.0 * correct / total
        scheduler.step()
        val_acc, val_loss = evaluate_model(model,val_loader, loss_fn, device)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_resnet18_fer.pth')
        print(f'Epoch [{epoch+1}/{epochs}], Loss:{train_loss/len(train_loader):.4f}, Train Acc: {train_accuracy:.2f}%, Val Acc: {val_acc:.2f}%, Val Loss: {val_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')
    print(f'Best Val Acc: {best_acc:.2f}%')
    return model

In [55]:
def evaluate_model(model, val_loader, loss_fn, device):
    correct = 0
    total = 0
    eval_loss = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            output = model(inputs)
            loss = loss_fn(output, labels)
            eval_loss += loss.item()
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100. *correct/total
    avg_loss = eval_loss / len(val_loader)
    return acc, avg_loss

In [56]:
def predict(model, test_loader, device):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for inputs, _ in test_loader:      # _ 表示忽略标签
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.append(predicted.cpu())
    return torch.cat(all_preds, dim=0)

In [57]:
model = model_train(model, device, train_loader, val_loader, optimizer, scheduler, 100)

Epoch [1/100], Loss:1.7916, Train Acc: 29.45%, Val Acc: 36.30%, Val Loss: 1.6311, LR: 0.001000
Epoch [2/100], Loss:1.5633, Train Acc: 39.23%, Val Acc: 40.57%, Val Loss: 1.5242, LR: 0.000999
Epoch [3/100], Loss:1.4421, Train Acc: 44.39%, Val Acc: 43.08%, Val Loss: 1.4853, LR: 0.000998
Epoch [4/100], Loss:1.3455, Train Acc: 48.71%, Val Acc: 44.92%, Val Loss: 1.4329, LR: 0.000996
Epoch [5/100], Loss:1.2656, Train Acc: 52.22%, Val Acc: 45.77%, Val Loss: 1.4306, LR: 0.000994
Epoch [6/100], Loss:1.1901, Train Acc: 55.46%, Val Acc: 46.76%, Val Loss: 1.4258, LR: 0.000991
Epoch [7/100], Loss:1.1200, Train Acc: 58.58%, Val Acc: 47.41%, Val Loss: 1.4171, LR: 0.000988
Epoch [8/100], Loss:1.0493, Train Acc: 61.32%, Val Acc: 48.00%, Val Loss: 1.4585, LR: 0.000984
Epoch [9/100], Loss:0.9776, Train Acc: 64.27%, Val Acc: 46.41%, Val Loss: 1.5011, LR: 0.000980
Epoch [10/100], Loss:0.9121, Train Acc: 66.89%, Val Acc: 45.76%, Val Loss: 1.5062, LR: 0.000976
Epoch [11/100], Loss:0.8443, Train Acc: 69.53%, V

In [59]:
model.load_state_dict(torch.load('best_resnet18_fer.pth'))
test_acc, test_loss = evaluate_model(model, test_loader, loss_fn, device)
print(f'Final Test Acc: {test_acc:.2f}%, Test Loss: {test_loss:.4f}')

Final Test Acc: 49.54%, Test Loss: 2.2769
